# Neural Network Training & Testing — Airline Sentiment Classifier
**Exercise PT-P2 — Audrey Benjamin Gutierrez**

This notebook implements the code-based (Keras/TensorFlow) counterpart to the no-code
classifier built in PT-P1. It follows the same overall pipeline used across all 20 logged
experiments (EXP-001 through EXP-15B): load data → build vocabulary → configure
hyperparameters → build the model → compile → train → evaluate → log → plot.

Each section below has a markdown cell explaining **what the following code cell does and
why**, followed by the code cell itself with inline comments on the individual functions
and parameters.

## 0. Mount Google Drive

Colab's runtime filesystem is temporary and separate from Google Drive. This cell mounts
your Drive so the dataset CSV (stored at `/content/drive/MyDrive/NN Training/`) is
accessible to `pandas` in the next steps. You'll be prompted to authorize access the first
time this runs in a session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # mounts your Google Drive at /content/drive


## 1. Imports & Reproducibility

Imports every library used downstream:
- **pandas / numpy** — data loading and array manipulation
- **tensorflow.keras** — model layers (`Embedding`, `GlobalAveragePooling1D`, `Dense`,
  `Dropout`), the `Sequential` model container, and both optimizers used across the 20
  experiments (`Adam`, `SGD`)
- **sklearn.model_selection.train_test_split** — creates the 70/15/15 train/validation/test
  split used in every experiment
- **sklearn.metrics** — accuracy, weighted F1, precision, and recall, computed the same way
  for both validation and test evaluation
- **Counter** — used later to build the vocabulary by word frequency
- **matplotlib** — renders the training/validation loss curves (Section III of the report)

The two `seed(42)` calls fix NumPy's and TensorFlow's random number generators. This is why
every experiment in the log uses `Random Seed: 42` — it keeps the data split and weight
initialization identical across runs, so that performance differences between experiments
reflect hyperparameter changes rather than random variation.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

from collections import Counter
import matplotlib.pyplot as plt

# Reproducibility: fixes the random data split and weight initialization
# so results are comparable across experiments (see Section II.A of the report).
np.random.seed(42)
tf.random.set_seed(42)


## 2. Load & Preprocess CSV Data

This cell performs the entire text-preprocessing pipeline described in Section II.B of the
report (Text Preprocessing):

1. **Load the CSV** into a DataFrame and inspect the first rows / column names to confirm
   the expected `Text` / `Label` schema loaded correctly.
2. **`build_vocab(texts, max_vocab_size)`** — lowercases and whitespace-tokenizes every
   training text, counts word frequency with `Counter`, and keeps only the
   `MAX_VOCAB_SIZE - 2` most common words (the `-2` reserves two special token IDs).
   Reserved tokens: `<PAD>` = 0 (used to pad short sequences) and `<UNK>` = 1 (used for any
   word not in the vocabulary — i.e. out-of-vocabulary handling).
3. **`text_to_sequence(text, vocab, max_len)`** — converts a single text string into a
   fixed-length list of integer token IDs: unknown words map to `<UNK>` (1), and the
   sequence is either zero-padded or truncated to exactly `MAX_LEN` tokens. This fixed
   length is required because Keras expects uniformly-shaped input batches.
4. **Encode all texts** and convert to NumPy arrays (`X_data`, `y_data`), then compute
   `num_classes` from the number of unique labels (3, in this study).
5. **70/15/15 split** — `train_test_split` is called twice: first splitting off 30% as a
   combined validation+test pool, then splitting that pool 50/50 into validation and test.
   This nested-split approach is how a 70/15/15 three-way split is done with a function that
   natively only supports two-way splits. The fixed `random_state=42` on both calls is what
   makes the split identical across every experiment in the log.

`MAX_LEN` and `MAX_VOCAB_SIZE` were varied across experiments (see Table I in the report,
columns MAX_LEN 20–50 and Vocab 2000–5000) — the values below reflect this run's
configuration.

In [ ]:
# ====================================
# 1. LOAD & PREPROCESS CSV DATA
# ====================================

CSV_FILEPATH = '/content/drive/MyDrive/NN Training/airline_sentiment_dataset.csv'

MAX_LEN = 40          # fixed sequence length: sequences are padded/truncated to this many tokens
MAX_VOCAB_SIZE = 3000  # cap on vocabulary size (most frequent words kept)

# Load CSV
df = pd.read_csv(CSV_FILEPATH)

print(df.head())       # sanity check: preview the first 5 rows
print(df.columns)      # sanity check: confirm expected column names (Text, Label, Category)

# ------------------------------------
# Vocabulary Builder
# ------------------------------------

def build_vocab(texts, max_vocab_size):
    # Lowercase + whitespace-tokenize every training text into a flat list of words
    words = [word.lower()
             for text in texts
             for word in str(text).split()]

    # Count frequency of every word across the training corpus
    word_counts = Counter(words)

    # Keep only the (max_vocab_size - 2) most frequent words;
    # -2 reserves slots for the <PAD> and <UNK> special tokens below
    most_common = word_counts.most_common(max_vocab_size - 2)

    # Assign each kept word an integer ID, starting at 2 (0 and 1 are reserved)
    vocab = {
        word: idx + 2
        for idx, (word, _) in enumerate(most_common)
    }

    vocab["<PAD>"] = 0  # padding token — fills unused positions in short sequences
    vocab["<UNK>"] = 1  # unknown/out-of-vocabulary token — used for words not in the vocab

    return vocab


def text_to_sequence(text, vocab, max_len):
    # Lowercase + tokenize the input text the same way as build_vocab
    tokens = str(text).lower().split()

    # Map each token to its vocab ID; unseen words fall back to <UNK> (1)
    seq = [vocab.get(token, 1) for token in tokens]

    # Force every sequence to exactly max_len tokens:
    # pad with <PAD> (0) if too short, truncate if too long
    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]

    return seq


# ------------------------------------
# Encode Text
# ------------------------------------

vocab = build_vocab(df['Text'].tolist(), MAX_VOCAB_SIZE)

encoded_texts = [
    text_to_sequence(txt, vocab, MAX_LEN)
    for txt in df['Text']
]

X_data = np.array(encoded_texts)       # shape: (num_examples, MAX_LEN)
y_data = np.array(df['Label'].values)  # integer class labels

num_classes = len(np.unique(y_data))   # 3 classes: Positive_Praise, Negative_Complaint, Urgent_Support

# ------------------------------------
# 70 / 15 / 15 Split
# ------------------------------------

# First split: 70% train, 30% temp (temp will become validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data,
    y_data,
    test_size=0.30,
    random_state=42
)

# Second split: divide the 30% temp pool evenly into validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)
print("Classes:", num_classes)
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))


## 3. Hyperparameter Configuration

This is the single cell edited between experiment runs — every one of the 20 logged
experiments (EXP-001 through EXP-15B) corresponds to a different set of values in this
`CONFIG` dictionary. The keys map directly onto the "Three Pillars" framework used to
organize the study (Section II.D of the report):

| CONFIG key | Pillar | What it controls |
|---|---|---|
| `embed_dim` | Architecture | Dimensionality of each word's learned embedding vector |
| `hidden_dim` | Architecture | Number of neurons in the Dense hidden layer |
| `dropout_rate` | Regularization | Fraction of hidden-layer units randomly dropped each training step |
| `learning_rate` | Optimization | Step size used by the optimizer during gradient descent |
| `optimizer_type` | Optimization | `"Adam"` or `"SGD"` — selects which optimizer is instantiated in Section 4 |
| `epochs` | Optimization | Number of full passes through the training data |
| `batch_size` | Optimization | Number of examples processed per gradient update |

`run_id` is a free-text label only — it does not affect training, but is used to tag printed
logs and the loss-curve plot title so results can be matched back to the tracking
spreadsheet.

In [ ]:
# ====================================
# 2. HYPERPARAMETER CONFIGURATION
# ====================================

CONFIG = {
    "run_id": "EXP-15- Batch Size Validation",  # label only, for logs/plot titles — does not affect training

    "embed_dim": 64,        # Architecture: embedding vector size per token

    "hidden_dim": 256,      # Architecture: neurons in the Dense hidden layer

    "dropout_rate": 0.4,    # Regularization: fraction of hidden units dropped each step

    "learning_rate": 0.0005,  # Optimization: optimizer step size

    "optimizer_type": "Adam",  # Optimization: "Adam" or "SGD" (selected in Section 4)

    "epochs": 40,            # Optimization: number of full passes over the training data

    "batch_size": 16          # Optimization: examples per gradient update
}


## 4. Keras Sequential Architecture

Builds the feedforward network used across every experiment in this study
(Section II.C of the report). Layer by layer:

1. **`Embedding`** — the model's feature-extraction entry point. Maps each integer token ID
   (0 to `len(vocab)-1`) to a learned dense vector of length `embed_dim`. `mask_zero=True`
   tells downstream layers to ignore `<PAD>` (token ID 0) positions rather than treating
   padding as real content. Output shape: `(batch, MAX_LEN, embed_dim)`.
2. **`GlobalAveragePooling1D`** — averages the embedding vectors across the (masked)
   sequence dimension, collapsing a variable-content sequence into a single fixed-length
   feature vector per example. Output shape: `(batch, embed_dim)`. This is the automated
   analogue to hand-engineered feature extraction referenced in Section IV.D of the report.
3. **`Dense(hidden_dim, activation='relu')`** — a fully-connected hidden layer that learns
   nonlinear combinations of the pooled embedding features.
4. **`Dropout(dropout_rate)`** — regularization: randomly zeroes out a fraction of the
   hidden layer's activations on each training step, reducing overfitting (see Section IV.B
   of the report for the EXP-003 → EXP-004 dropout comparison).
5. **`Dense(num_classes, activation='softmax')`** — the output layer, producing a
   probability distribution over the 3 sentiment/intent classes.

`model.summary()` prints the resulting layer-by-layer architecture and parameter counts.

In [ ]:
# ====================================
# 3. KERAS SEQUENTIAL ARCHITECTURE
# ====================================

model = Sequential()

model.add(
    Embedding(
        input_dim=len(vocab),          # vocabulary size (rows in the embedding lookup table)
        output_dim=CONFIG["embed_dim"], # length of each token's embedding vector
        input_length=MAX_LEN,           # fixed input sequence length
        mask_zero=True                  # ignore <PAD> (token ID 0) in downstream layers
    )
)

model.add(GlobalAveragePooling1D())  # averages token embeddings into one feature vector per example

model.add(
    Dense(
        CONFIG["hidden_dim"],
        activation='relu'   # ReLU introduces nonlinearity, letting the model learn complex patterns
    )
)

model.add(
    Dropout(
        CONFIG["dropout_rate"]  # regularization: randomly drops this fraction of units each step
    )
)

model.add(
    Dense(
        num_classes,
        activation='softmax'  # converts raw outputs into class probabilities summing to 1
    )
)

model.summary()  # prints the architecture: layer types, output shapes, and parameter counts


## 5. Optimizer & Compilation

Two things happen here, both referenced in the pipeline-mapping discussion (Section IV.D of
the report):

1. **Optimizer selection** — an `if/else` on `CONFIG["optimizer_type"]` instantiates either
   `Adam` or `SGD` with the configured `learning_rate`. Across the 20 experiments, Adam
   consistently outperformed SGD by a wide margin (see EXP-08A/EXP-08B in Section IV.B of
   the report).
2. **`model.compile()`** — wires the model to its loss function, optimizer, and tracked
   metric:
   - `loss='sparse_categorical_crossentropy'` — appropriate for integer-encoded (rather than
     one-hot) multi-class labels; this is the loss function minimized during training.
   - `optimizer=opt` — the Adam/SGD instance created above.
   - `metrics=['accuracy']` — accuracy is tracked and printed during training in addition to
     loss, though the report's primary comparison metric is weighted F1 (computed separately
     in Sections 7–8 below).

Compiling does not train the model — it only configures *how* training will happen once
`model.fit()` is called in the next cell.

In [ ]:
# ====================================
# 4. OPTIMIZER & COMPILATION
# ====================================

if CONFIG["optimizer_type"] == "Adam":
    opt = Adam(
        learning_rate=CONFIG["learning_rate"]
    )
else:
    opt = SGD(
        learning_rate=CONFIG["learning_rate"]
    )

model.compile(
    loss='sparse_categorical_crossentropy',  # matches integer class labels (not one-hot)
    optimizer=opt,                            # Adam or SGD instance configured above
    metrics=['accuracy']                      # tracked and printed alongside loss during training
)

model.summary()


## 6. Training

This is where the forward pass, loss calculation, and backward pass (weight updates) all
execute — internally, on every batch, for every epoch — via Keras's `model.fit()`
(Section IV.D of the report). None of the gradient computation is written manually; Keras's
automatic differentiation computes gradients of the loss with respect to every trainable
parameter and applies the optimizer's update rule.

Arguments:
- `X_train, y_train` — the training data
- `validation_data=(X_val, y_val)` — evaluated after every epoch (not used for training
  itself) so `history` records both training and validation loss/accuracy per epoch — this
  is the data plotted in Section 6 (Loss Curve) below and analyzed throughout Section III of
  the report.
- `epochs`, `batch_size` — from `CONFIG`.
- `verbose=1` — prints a progress bar and per-epoch metrics to the console.

`training_time` is logged in minutes for the tracking spreadsheet's "Training Time (min)"
column.

In [ ]:
# ====================================
# 5. TRAINING
# ====================================

print(f"Starting Training Run: {CONFIG['run_id']}")

import time

start_time = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),  # evaluated each epoch; not used to update weights
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    verbose=1
)

training_time = round((time.time() - start_time) / 60, 2)

print("Training Time (min):", training_time)


## 7. Validation Metrics

Computes the metrics used to compare experiments in the report's results tables (Table II
and Table III):

1. **Predict** on the validation set (`model.predict`) — returns class *probabilities*, not
   labels, so `np.argmax(..., axis=1)` converts each prediction to the single most-likely
   class.
2. **Compute metrics** — `accuracy_score`, `precision_score`, `recall_score`, and `f1_score`,
   all using `average='weighted'` so that each class's score is weighted by how many true
   examples it has (appropriate here since the dataset can be imbalanced across categories).
   `zero_division=0` prevents a warning/crash if a class has no predicted examples.
3. **Pull final-epoch loss values** directly from `history` (populated during `model.fit()`
   in the previous cell) rather than recomputing them.
4. **Print an experiment log entry** — this is the exact console output that was
   screenshotted and used as source evidence for every experiment row in the report's Table
   I and Table II.

In [ ]:
# ====================================
# VALIDATION METRICS
# ====================================

val_pred_probs = model.predict(X_val)          # shape: (n_val, num_classes) — class probabilities
val_preds = np.argmax(val_pred_probs, axis=1)   # convert probabilities to predicted class labels

val_accuracy = accuracy_score(y_val, val_preds)
val_precision = precision_score(
    y_val,
    val_preds,
    average='weighted',  # weight each class's score by its support (number of true examples)
    zero_division=0       # avoid divide-by-zero warnings if a class has no predictions
)

val_recall = recall_score(
    y_val,
    val_preds,
    average='weighted',
    zero_division=0
)

val_f1 = f1_score(
    y_val,
    val_preds,
    average='weighted'
)

train_loss = history.history['loss'][-1]      # final-epoch training loss
val_loss = history.history['val_loss'][-1]    # final-epoch validation loss

train_accuracy = history.history['accuracy'][-1] * 100
val_accuracy_percent = val_accuracy * 100

print("\n===== EXP LOG ENTRY =====")

print("Run ID:", CONFIG["run_id"])
print("Learning Rate:", CONFIG["learning_rate"])
print("Hidden Dim:", CONFIG["hidden_dim"])
print("Dropout:", CONFIG["dropout_rate"])

print("\nTraining Loss:", round(train_loss,4))
print("Validation Loss:", round(val_loss,4))

print("Train Accuracy (%):", round(train_accuracy,2))
print("Validation Accuracy (%):", round(val_accuracy_percent,2))

print("Validation Precision:", round(val_precision,4))
print("Validation Recall:", round(val_recall,4))
print("Validation Weighted F1:", round(val_f1,4))


## 8. Test Evaluation

Repeats the same prediction-and-scoring pattern as the previous cell, but on the held-out
**test set** (`X_test`, `y_test`) — data the model has never seen during training *or*
validation-based tuning.

This is the most important cell for the report's central finding (Section III.C /
Section IV.C): while validation metrics were used to select the "best" configuration during
tuning, several configurations tied on validation weighted F1 (0.9051–0.9076) but produced
meaningfully different **test** F1-scores (0.5281–0.8296). Test-set performance, computed
here, is what actually determines which configuration generalizes best — it's why EXP-11
was selected as the champion model over EXP-09, despite both scoring identically on
validation.

In [ ]:
# ====================================
# TEST EVALUATION
# ====================================

test_probs = model.predict(X_test)

test_preds = np.argmax(
    test_probs,
    axis=1
)

test_acc = accuracy_score(
    y_test,
    test_preds
)

test_precision = precision_score(
    y_test,
    test_preds,
    average='weighted',
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_preds,
    average='weighted',
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_preds,
    average='weighted'
)

print("\n===== TEST RESULTS =====")

print("Test Accuracy:", round(test_acc,4))
print("Test Precision:", round(test_precision,4))
print("Test Recall:", round(test_recall,4))
print("Test F1:", round(test_f1,4))


## 9. Consolidated Training Log

Prints every field needed for one row of the tracking spreadsheet
(`FFBP_Training_Log_Template.xlsx`) in one place — hyperparameters (from `CONFIG`), loss and
accuracy (from Sections 7–8), and run metadata (training time, random seed). This cell's
output is what gets copied into the spreadsheet after each experiment, rather than manually
re-typing values scattered across the earlier cells' outputs.

Note: the `Precision`/`Recall`/`F1 Score` lines here use the **test** set values
(`test_precision`, `test_recall`, `test_f1`) computed in Section 8, not the validation values
from Section 7 — worth double-checking against the spreadsheet column headers to confirm
which metric each column is meant to hold before logging.

In [ ]:
print("\n========== TRAINING LOG ==========")

print(f"Run ID: {CONFIG['run_id']}")
print(f"Dataset: Airline Sentiment Dataset")
print(f"Input Features: {MAX_LEN}")          # sequence length (MAX_LEN), not vocabulary size
print(f"Hidden Layers: 1")
print(f"Neurons per Layer: {CONFIG['hidden_dim']}")
print(f"Output Neurons: {num_classes}")
print(f"Activation Function: ReLU")
print(f"Learning Rate: {CONFIG['learning_rate']}")
print(f"Optimizer: {CONFIG['optimizer_type']}")
print(f"Epochs: {CONFIG['epochs']}")
print(f"Batch Size: {CONFIG['batch_size']}")

print(f"Train Loss: {round(train_loss,4)}")
print(f"Validation Loss: {round(val_loss,4)}")

print(f"Train Accuracy (%): {round(train_accuracy,2)}")
print(f"Validation Accuracy (%): {round(val_accuracy_percent,2)}")

print(f"Precision: {round(test_precision,4)}")  # test-set precision (Section 8)
print(f"Recall: {round(test_recall,4)}")        # test-set recall (Section 8)
print(f"F1 Score: {round(test_f1,4)}")          # test-set F1 (Section 8)

print(f"Training Time (min): {training_time}")
print(f"Random Seed: 42")


## 10. Loss Curve

Plots training loss and validation loss against epoch number, using the full per-epoch
history recorded by `model.fit()` in Section 5 (`history.history['loss']` and
`history.history['val_loss']`) — not just the final-epoch values used in the log entries
above.

This plot is the primary diagnostic tool used throughout Section III.B of the report to
classify each experiment's fit:
- **Underfitting** — both curves stay high and close together (e.g., EXP-002, EXP-08A).
- **Overfitting** — training loss keeps falling while validation loss rises or diverges
  (e.g., EXP-003).
- **Good fit** — both curves decrease and stabilize close together (e.g., EXP-11).

The plot title is pulled from `CONFIG['run_id']`, which is why each experiment's saved plot
image is self-labeled (e.g., "Loss Curves (EXP-09-VOCAB-TUNED)") — this is what makes the
extracted screenshots traceable back to their originating configuration when assembling the
report's Figures 2–7.

In [ ]:
# ====================================
# 6. LOSS CURVE
# ====================================

train_losses = history.history['loss']       # per-epoch training loss
val_losses = history.history['val_loss']     # per-epoch validation loss

epochs_range = range(
    1,
    CONFIG["epochs"] + 1
)

plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    train_losses,
    label='Training Loss',
    linewidth=2,
    marker='o'
)

plt.plot(
    epochs_range,
    val_losses,
    label='Validation Loss',
    linewidth=2,
    linestyle='--',
    marker='s'
)

plt.title(
    f"Loss Curves ({CONFIG['run_id']})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()
plt.grid(True)

plt.show()
